In [1]:
import numpy as np
import logging
from sklearn.datasets import make_regression, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Configure logging
logging.basicConfig(
    level=logging.INFO,  # Set log level to INFO
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Function to train and evaluate LassoCV
def train_evaluate_lasso(X_train, X_test, y_train, y_test):
    logger.info("Training LassoCV...")
    lasso = LassoCV(cv=5, random_state=42)
    lasso.fit(X_train, y_train)
    logger.info("LassoCV training completed.")
    
    logger.info("Evaluating LassoCV...")
    y_pred = lasso.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    logger.info(f"LassoCV evaluation completed. MSE: {mse:.4f}, R²: {r2:.4f}")
    return mse, r2

# Function to train and evaluate AdaptiveLasso
def train_evaluate_adaptive_lasso(X_train, X_test, y_train, y_test):
    logger.info("Training AdaptiveLasso...")
    adaptivelasso = MAPAdaptiveLasso(lam_prior='adaptive', lik='normal', max_iters=300, track=False)
    adaptivelasso.fit(X_train, y_train)
    logger.info("AdaptiveLasso training completed.")
    
    logger.info("Evaluating AdaptiveLasso...")
    y_pred = adaptivelasso.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    logger.info(f"AdaptiveLasso evaluation completed. MSE: {mse:.4f}, R²: {r2:.4f}")
    return mse, r2

# Function to compare both methods
def compare_lasso_methods(X, y, dataset_name):
    logger.info(f"Starting comparison for dataset: {dataset_name}")
    
    # Split data into train and test sets
    logger.info("Splitting data into train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    logger.info("Data split completed.")
    
    # Train and evaluate LassoCV
    mse_lasso, r2_lasso = train_evaluate_lasso(X_train, X_test, y_train, y_test)

    # Train and evaluate AdaptiveLasso
    mse_adaptive, r2_adaptive = train_evaluate_adaptive_lasso(X_train, X_test, y_train, y_test)

    # Print results
    logger.info(f"Comparison results for dataset: {dataset_name}")
    logger.info(f"LassoCV - MSE: {mse_lasso:.4f}, R²: {r2_lasso:.4f}")
    logger.info(f"AdaptiveLasso - MSE: {mse_adaptive:.4f}, R²: {r2_adaptive:.4f}")

# Synthetic Dataset
logger.info("Generating synthetic dataset...")
X_synthetic, y_synthetic = make_regression(
    n_samples=1000, n_features=20, n_informative=5, noise=0.1, random_state=42
)
logger.info("Synthetic dataset generated.")
compare_lasso_methods(X_synthetic, y_synthetic, "Synthetic Dataset")

# Diabetes Dataset
logger.info("Loading diabetes dataset...")
diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target
logger.info("Diabetes dataset loaded.")
compare_lasso_methods(X_diabetes, y_diabetes, "Diabetes Dataset")

2025-06-18 22:25:42,535 - INFO - Generating synthetic dataset...
2025-06-18 22:25:42,537 - INFO - Synthetic dataset generated.
2025-06-18 22:25:42,537 - INFO - Starting comparison for dataset: Synthetic Dataset
2025-06-18 22:25:42,537 - INFO - Splitting data into train and test sets...
2025-06-18 22:25:42,538 - INFO - Data split completed.
2025-06-18 22:25:42,538 - INFO - Training LassoCV...
2025-06-18 22:25:42,558 - INFO - LassoCV training completed.
2025-06-18 22:25:42,558 - INFO - Evaluating LassoCV...
2025-06-18 22:25:42,559 - INFO - LassoCV evaluation completed. MSE: 0.0489, R²: 1.0000
2025-06-18 22:25:42,559 - INFO - Training AdaptiveLasso...
INFO:2025-06-18 22:25:42,586:jax._src.xla_bridge:867: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/u543860/miniforge3/envs/arm_jax_env/bin/../lib/libtpu.so' (no such file), '/usr/lib

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from adaptivelasso.skcompat import MAPAdaptiveLasso
from sklearn.metrics import mean_squared_error, r2_score

# Define parameter grids
max_iters_list = [300, 1000, 3000]
lr_pre_list = [0.001, 0.01, 0.1]

# Prepate data
X_synthetic, y_synthetic = make_regression(
    n_samples=1000, n_features=20, n_informative=5, noise=0.1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X_synthetic, y_synthetic, test_size=0.2, random_state=42)

results = []
for max_iters in max_iters_list:
    for lr_pre in lr_pre_list:
        print(f"Testing max_iters={max_iters}, lr_pre={lr_pre}")
        pipe = Pipeline([
            ('scale', StandardScaler()),
            ('jax_model', MAPAdaptiveLasso(
                lam_prior='adaptive',
                lik='normal',
                max_iters=max_iters,
                lr_pre=lr_pre,
                track=False
            ))
        ])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"  MSE: {mse:.4f}, R²: {r2:.4f}")
        results.append({'max_iters': max_iters, 'lr_pre': lr_pre, 'mse': mse, 'r2': r2})

# Optionally, display all results as a table
import pandas as pd
print(pd.DataFrame(results))